# RAG Pipeline Architecture
<pre>
Documents (PDF/TXT)
        ↓
Document Loader
        ↓
Text Chunking
        ↓
GPU Embedding Model
        ↓
Vector Database (FAISS / Chroma)
        ↓
Retriever
        ↓
LLM (Llama / Mistral / OpenAI)
        ↓
Answer
</pre>

# Loading Libraries

In [ ]:
import os
import faiss
import numpy as np

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer
from openai import OpenAI


# OpenRouter Configuration
# Arcee AI: Trinity Large Preview(LLM)

In [ ]:
OPENROUTER_API_KEY = "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"

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)


#Load Documents

In [ ]:
def load_documents(pdf_path):

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    return documents

#Split Documents

In [ ]:
def split_documents(documents):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = text_splitter.split_documents(documents)

    return chunks

# Generate Embeddings (GPU)

In [ ]:
def create_embeddings(chunks):

    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cuda"
    )

    texts = [chunk.page_content for chunk in chunks]

    embeddings = model.encode(
        texts,
        convert_to_tensor=True
    )

    return model, embeddings, texts

# Store in FAISS Vector Database

In [ ]:
def build_faiss_index(embeddings):

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings.cpu().detach().numpy())

    return index

# Retrieval Function

In [ ]:
def retrieve(query, model, index, texts, top_k=3):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    retrieved_docs = [texts[i] for i in indices[0]]

    return retrieved_docs

# OpenRouter LLM Answer

In [ ]:
def generate_answer(query, context):

    context_text = "\n".join(context)

    prompt = f"""
Use the following context to answer the question.

Context:
{context_text}

Question:
{query}

Answer:
"""

    completion = client.chat.completions.create(
        model="meta-llama/llama-3-8b-instruct",
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=300
    )

    return completion.choices[0].message.content

# Main RAG Pipeline

In [ ]:
def rag_pipeline(pdf_path):

    print("Loading document...")
    documents = load_documents(pdf_path)

    print("Splitting document...")
    chunks = split_documents(documents)

    print("Generating embeddings...")
    model, embeddings, texts = create_embeddings(chunks)

    print("Building FAISS index...")
    index = build_faiss_index(embeddings)

    print("RAG system ready!")

    while True:

        query = input("\nAsk a question (type 'exit' to quit): ")

        if query.lower() == "exit":
            break

        context = retrieve(query, model, index, texts)

        answer = generate_answer(query, context)

        print("\nAnswer:\n")
        print(answer)

# Run

In [ ]:
if __name__ == "__main__":

    pdf_file = "/content/drive/MyDrive/Colab Notebooks/attention-is-all-you-need-Paper.pdf"

    rag_pipeline(pdf_file)

Loading document...
Splitting document...
Generating embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index...
RAG system ready!

Ask a question (type 'exit' to quit): What is the transformer architecture?

Answer:

The Transformer architecture is a model that uses stacked self-attention and point-wise, fully connected feed-forward networks in both the encoder and decoder, with residual connections and layer normalization.

Ask a question (type 'exit' to quit): Why is self-attention used?

Answer:

According to the context, self-attention is used to relate different positions of a single sequence in order to compute a representation of the sequence, which has been successfully applied to various tasks such as reading comprehension, abstractive summarization, textual entailment, and learning task-independent sentence representations.

Ask a question (type 'exit' to quit): exit
